In [ ]:
# ================== MAPA DESDE MATRIZ PRECALCULADA CON TÍTULO ESTÉTICO ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
from folium.plugins import MarkerCluster
from shapely.geometry import MultiPoint, Point
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# Buscar archivos Excel en la carpeta de inputs
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Inputs_mapa'
archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))

if archivos_excel:
    archivo_matriz = archivos_excel[0]  
    print(f"Archivo seleccionado: {os.path.basename(archivo_matriz)}")
else:
    print("No se encontraron archivos Excel en la carpeta de inputs")
    archivo_matriz = None

output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
os.makedirs(output_dir, exist_ok=True)
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')

# Leer la hoja de recorridos
df_recorridos = pd.read_excel(archivo_matriz, sheet_name='Recorridos')

# Definición de columnas
col_met = '🏠 MET'
col_ruta = '🛣️ Ruta'
col_secuencia = '🔢 Secuencia'
col_tipo = 'Tipo'
col_codigo = '🆔 Codigo'
col_longitud = '🌍 Longitud'
col_latitud = '🌍 Latitud'
col_nombre = '📚 Nombre'
col_distancia_sig = '📏 Distancia_al_siguiente_km'

mets = sorted(df_recorridos[col_met].dropna().unique())
rutas = sorted(df_recorridos[col_ruta].dropna().unique())

# Widgets de interfaz
met_selector = widgets.SelectMultiple(options=mets, value=tuple(mets), description='Selecciona MET(s):', layout=widgets.Layout(width='50%'))
ruta_selector = widgets.SelectMultiple(options=rutas, value=tuple(rutas), description='Selecciona Ruta(s):', layout=widgets.Layout(width='50%'))
generar_btn = widgets.Button(description='Generar mapa avanzado', button_style='success', layout=widgets.Layout(width='300px'))
output_map = widgets.Output()

def actualizar_rutas(*args):
    mets_seleccionados = list(met_selector.value)
    if mets_seleccionados:
        rutas_filtradas = sorted(df_recorridos[df_recorridos[col_met].isin(mets_seleccionados)][col_ruta].dropna().unique())
    else:
        rutas_filtradas = sorted(df_recorridos[col_ruta].dropna().unique())
    ruta_selector.options = rutas_filtradas

met_selector.observe(actualizar_rutas, names='value')
actualizar_rutas()

def generar_mapa(b):
    with output_map:
        clear_output()
        mets_sel = list(met_selector.value)
        rutas_sel = list(ruta_selector.value)
        
        if not mets_sel or not rutas_sel:
            print('Selecciona METs y Rutas primero.')
            return

        # Filtrar datos para estadísticas del título
        df_filtrado = df_recorridos[(df_recorridos[col_met].isin(mets_sel)) & (df_recorridos[col_ruta].isin(rutas_sel))]
        total_clientes = df_filtrado[df_filtrado[col_tipo] == 'Cliente'].shape[0]

        # Inicializar mapa
        met_row_init = df_recorridos[df_recorridos[col_met] == mets_sel[0]].iloc[0]
        mapa = folium.Map(location=[met_row_init[col_latitud], met_row_init[col_longitud]], zoom_start=10)

        colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'black']
        color_idx = 0

        # Creación de capas por combinación
        for met in mets_sel:
            met_clean = str(met).replace('MET ', '').strip()
            df_met = df_recorridos[(df_recorridos[col_met] == met) & (df_recorridos[col_ruta].isin(rutas_sel))]
            
            for ruta in df_met[col_ruta].unique():
                capa_nombre = f"MET {met_clean} - Ruta {ruta}"
                fg = folium.FeatureGroup(name=capa_nombre)
                
                df_ruta = df_met[df_met[col_ruta] == ruta].sort_values(col_secuencia)
                coords = list(zip(df_ruta[col_latitud], df_ruta[col_longitud]))
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1

                folium.PolyLine(coords, color=color_ruta, weight=5, opacity=0.7).add_to(fg)

                for _, row in df_ruta.iterrows():
                    if row[col_tipo] == 'MET':
                        icon = CustomIcon(icon_met_path, icon_size=(32,32)) if os.path.exists(icon_met_path) else folium.Icon(color='black', icon='home')
                    else:
                        icon = DivIcon(html=f'<div style="font-size:12px; color:white; background:{color_ruta}; border-radius:50%; width:20px; height:20px; text-align:center; line-height:20px; border:1px solid #fff;">{row[col_secuencia]}</div>')
                    
                    folium.Marker([row[col_latitud], row[col_longitud]], icon=icon, popup=row[col_nombre]).add_to(fg)
                
                fg.add_to(mapa)

        # 1. INYECCIÓN DEL TÍTULO ESTÉTICO DINÁMICO
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 2px solid #fff; min-width: 450px;">
            <h2 style="margin: 0; text-align: center; font-size:16px; text-shadow: 1px 1px 2px rgba(0,0,0,0.5); font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
                🗺️ ANÁLISIS COMERCIAL - VOLUMEN E IMPORTE
            </h2>
            <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.95; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
                📍 METs: <b>{len(mets_sel)}</b> | 🛣️ Rutas: <b>{len(rutas_sel)}</b> | 👥 Clientes: <b>{total_clientes}</b><br>
                🛣️ Vista por Recorridos: Líneas y Secuencias | 📦 Marcadores Numerados | 🎨 Colores por Ruta
            </p>
        </div>
        '''
        mapa.get_root().html.add_child(folium.Element(titulo_html))

        # 2. INYECCIÓN DE CSS (Controles de capas con botones)
        scroll_layers_css = '''
        <style>
        .leaflet-control-layers-overlays { display: none !important; }
        .leaflet-control-layers-list { max-height: 400px; overflow-y: auto; min-width: 220px; }
        .layer-control-buttons { padding: 8px; border-bottom: 1px solid #ddd; background: #f8f8f8; display: flex; gap: 5px; }
        .layer-btn { padding: 5px; font-size: 11px; cursor: pointer; border-radius: 4px; flex: 1; font-weight: bold; border: 1px solid #ccc; }
        .layer-btn.select-all { background: #4CAF50; color: white; }
        .layer-btn.deselect-all { background: #f44336; color: white; }
        .met-buttons-row { padding: 6px; border-bottom: 1px solid #ddd; background: #e3f2fd; display: flex; gap: 4px; flex-wrap: wrap; }
        .met-btn { padding: 4px; font-size: 10px; border: 1px solid #2196F3; background: white; color: #1976D2; cursor: pointer; border-radius: 4px; flex: 1; min-width: 65px; text-align: center; }
        .met-btn:hover { background: #2196F3; color: white; }
        </style>
        '''
        mapa.get_root().html.add_child(folium.Element(scroll_layers_css))

        # 3. INYECCIÓN DE JS (Lógica de botones)
        layer_control_js = '''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            setTimeout(function() {
                const layerControl = document.querySelector('.leaflet-control-layers');
                const layersList = layerControl ? layerControl.querySelector('.leaflet-control-layers-list') : null;
                const overlaysDiv = layerControl ? layerControl.querySelector('.leaflet-control-layers-overlays') : null;
                
                if (overlaysDiv && layersList) {
                    const buttonsDiv = document.createElement('div');
                    buttonsDiv.className = 'layer-control-buttons';
                    buttonsDiv.innerHTML = '<button class="layer-btn select-all">✓ Todo</button><button class="layer-btn deselect-all">✗ Nada</button>';
                    
                    const metButtonsDiv = document.createElement('div');
                    metButtonsDiv.className = 'met-buttons-row';
                    
                    const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                    const mets = new Set();
                    checkboxes.forEach(cb => {
                        const label = cb.parentElement.querySelector('span').textContent.trim();
                        const match = label.match(/^MET\\s+([^\\s-]+)/);
                        if (match) mets.add(match[1]);
                    });

                    mets.forEach(met => {
                        const btn = document.createElement('button');
                        btn.className = 'met-btn';
                        btn.textContent = met;
                        btn.onclick = function() {
                            checkboxes.forEach(cb => {
                                const label = cb.parentElement.querySelector('span').textContent.trim();
                                if (label.includes('MET ' + met)) { if(!cb.checked) cb.click(); }
                                else { if(cb.checked) cb.click(); }
                            });
                        };
                        metButtonsDiv.appendChild(btn);
                    });

                    layersList.insertBefore(buttonsDiv, overlaysDiv);
                    layersList.insertBefore(metButtonsDiv, overlaysDiv);

                    buttonsDiv.querySelector('.select-all').onclick = () => { checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); };
                    buttonsDiv.querySelector('.deselect-all').onclick = () => { checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); };
                }
            }, 1500);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(layer_control_js))

        folium.LayerControl(collapsed=False).add_to(mapa)
        
        nombre_mapa = os.path.join(output_dir, f'mapa_final_{fecha_hora}.html')
        mapa.save(nombre_mapa)
        print(f'✅ Mapa generado: {nombre_mapa}')

generar_btn.on_click(generar_mapa)
display(widgets.VBox([met_selector, ruta_selector, generar_btn, output_map]))

In [ ]:
# ================== MAPA CON FILTROS POR MET Y POR RUTA ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuración de rutas
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Inputs_mapa'
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
os.makedirs(output_dir, exist_ok=True)

archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))
archivo_matriz = archivos_excel[0] if archivos_excel else None

# Leer datos
df_recorridos = pd.read_excel(archivo_matriz, sheet_name='Recorridos')

# Columnas
col_met, col_ruta, col_secuencia = '🏠 MET', '🛣️ Ruta', '🔢 Secuencia'
col_tipo, col_nombre = 'Tipo', '📚 Nombre'
col_lat, col_lon = '🌍 Latitud', '🌍 Longitud'

mets = sorted(df_recorridos[col_met].dropna().unique())
rutas = sorted(df_recorridos[col_ruta].dropna().unique())

# Widgets
met_selector = widgets.SelectMultiple(options=mets, value=tuple(mets), description='METs:', layout=widgets.Layout(width='45%'))
ruta_selector = widgets.SelectMultiple(options=rutas, value=tuple(rutas), description='Rutas:', layout=widgets.Layout(width='45%'))
generar_btn = widgets.Button(description='Generar Mapa con Filtros', button_style='success', layout=widgets.Layout(width='300px'))
output_map = widgets.Output()

def generar_mapa(b):
    with output_map:
        clear_output()
        mets_sel, rutas_sel = list(met_selector.value), list(ruta_selector.value)
        if not mets_sel or not rutas_sel: return
        
        df_filtrado = df_recorridos[(df_recorridos[col_met].isin(mets_sel)) & (df_recorridos[col_ruta].isin(rutas_sel))]
        mapa = folium.Map(location=[df_filtrado[col_lat].mean(), df_filtrado[col_lon].mean()], zoom_start=10)
        colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'black']
        color_idx = 0

        for met in mets_sel:
            met_clean = str(met).replace('MET ', '').strip()
            df_met = df_filtrado[df_filtrado[col_met] == met]
            for ruta in df_met[col_ruta].unique():
                capa_nombre = f"MET {met_clean} - Ruta {ruta}"
                fg = folium.FeatureGroup(name=capa_nombre)
                df_ruta = df_met[df_met[col_ruta] == ruta].sort_values(col_secuencia)
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                
                folium.PolyLine(list(zip(df_ruta[col_lat], df_ruta[col_longitud])), color=color_ruta, weight=5, opacity=0.7).add_to(fg)
                for _, row in df_ruta.iterrows():
                    if row[col_tipo] == 'MET':
                        icon = CustomIcon(icon_met_path, icon_size=(32,32)) if os.path.exists(icon_met_path) else folium.Icon(color='black')
                    else:
                        icon = DivIcon(html=f'<div style="font-size:11px; color:white; background:{color_ruta}; border-radius:50%; width:18px; height:18px; text-align:center; line-height:18px; border:1px solid #fff;">{row[col_secuencia]}</div>')
                    folium.Marker([row[col_lat], row[col_lon]], icon=icon, popup=row[col_nombre]).add_to(fg)
                fg.add_to(mapa)

        # TÍTULO ESTÉTICO
        mapa.get_root().html.add_child(folium.Element(f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 2px solid #fff; min-width: 450px; font-family: sans-serif;">
            <h2 style="margin: 0; text-align: center; font-size:16px;">🗺️ ANÁLISIS COMERCIAL</h2>
            <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px;">
                📍 METs: <b>{len(mets_sel)}</b> | 🛣️ Rutas: <b>{len(rutas_sel)}</b> | 👥 Clientes: <b>{df_filtrado[df_filtrado[col_tipo]=='Cliente'].shape[0]}</b>
            </p>
        </div>
        '''))

        # CSS PARA BOTONES
        mapa.get_root().html.add_child(folium.Element('''
        <style>
        .leaflet-control-layers-overlays { display: none !important; }
        .leaflet-control-layers-list { max-height: 450px; overflow-y: auto; min-width: 250px; font-family: sans-serif; }
        .layer-section-title { font-size: 10px; font-weight: bold; color: #666; padding: 4px 8px; text-transform: uppercase; background: #eee; }
        .layer-control-buttons, .met-buttons-row, .ruta-buttons-row { padding: 6px; display: flex; gap: 4px; flex-wrap: wrap; border-bottom: 1px solid #ddd; }
        .layer-btn, .met-btn, .ruta-btn { padding: 4px; font-size: 10px; cursor: pointer; border-radius: 4px; border: 1px solid #ccc; background: white; flex: 1; min-width: 45px; text-align: center; }
        .select-all { background: #4CAF50; color: white; } .deselect-all { background: #f44336; color: white; }
        .met-btn { border-color: #2196F3; color: #1976D2; } .ruta-btn { border-color: #ff9800; color: #e65100; }
        .met-btn:hover { background: #2196F3; color: white; } .ruta-btn:hover { background: #ff9800; color: white; }
        </style>
        '''))

        # JS PARA INYECTAR FILTROS DE MET Y RUTA
        layer_control_js = '''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            setTimeout(function() {
                const layersList = document.querySelector('.leaflet-control-layers-list');
                const overlaysDiv = document.querySelector('.leaflet-control-layers-overlays');
                if (!overlaysDiv) return;
                
                const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                const mets = new Set(); const rutas = new Set();
                
                checkboxes.forEach(cb => {
                    const txt = cb.parentElement.querySelector('span').textContent;
                    const matchMet = txt.match(/MET ([^\\s-]+)/);
                    const matchRuta = txt.match(/Ruta ([^\\s-]+)/);
                    if(matchMet) mets.add(matchMet[1]);
                    if(matchRuta) rutas.add(matchRuta[1]);
                });

                const container = document.createElement('div');
                
                // 1. GESTIÓN GENERAL
                container.innerHTML += '<div class="layer-section-title">General</div><div class="layer-control-buttons"><button class="layer-btn select-all">✓ Todo</button><button class="layer-btn deselect-all">✗ Nada</button></div>';
                
                // 2. FILTRO POR MET
                let metHtml = '<div class="layer-section-title">Filtrar por MET</div><div class="met-buttons-row">';
                Array.from(mets).sort().forEach(m => metHtml += `<button class="met-btn" data-met="${m}">${m}</button>`);
                container.innerHTML += metHtml + '</div>';

                // 3. FILTRO POR RUTA (Multi-MET)
                let rutaHtml = '<div class="layer-section-title">Filtrar por N° Ruta</div><div class="ruta-buttons-row">';
                Array.from(rutas).sort((a,b) => a-b).forEach(r => rutaHtml += `<button class="ruta-btn" data-ruta="${r}">R${r}</button>`);
                container.innerHTML += rutaHtml + '</div>';

                layersList.insertBefore(container, overlaysDiv);

                // Lógica
                container.addEventListener('click', (e) => {
                    if(!e.target.classList.contains('layer-btn') && !e.target.classList.contains('met-btn') && !e.target.classList.contains('ruta-btn')) return;
                    
                    const isAll = e.target.classList.contains('select-all');
                    const isNone = e.target.classList.contains('deselect-all');
                    const targetMet = e.target.getAttribute('data-met');
                    const targetRuta = e.target.getAttribute('data-ruta');

                    checkboxes.forEach(cb => {
                        const txt = cb.parentElement.querySelector('span').textContent;
                        if(isAll) { if(!cb.checked) cb.click(); }
                        else if(isNone) { if(cb.checked) cb.click(); }
                        else if(targetMet) {
                            const match = txt.includes('MET ' + targetMet);
                            if(match && !cb.checked) cb.click(); else if(!match && cb.checked) cb.click();
                        }
                        else if(targetRuta) {
                            const match = txt.endsWith('Ruta ' + targetRuta);
                            if(match && !cb.checked) cb.click(); else if(!match && cb.checked) cb.click();
                        }
                    });
                });
            }, 1500);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(layer_control_js))
        folium.LayerControl(collapsed=False).add_to(mapa)
        
        path_html = os.path.join(output_dir, f'mapa_filtros_{datetime.now().strftime("%H%M%S")}.html')
        mapa.save(path_html)
        print(f'✅ Guardado en: {path_html}')

generar_btn.on_click(generar_mapa)
display(widgets.VBox([met_selector, ruta_selector, generar_btn, output_map]))

In [ ]:
# ================== MAPA CON FILTROS OPTIMIZADOS (4 POR FILA + SCROLL) ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuración de rutas
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Inputs_mapa'
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
os.makedirs(output_dir, exist_ok=True)

archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))
archivo_matriz = archivos_excel[0] if archivos_excel else None

# Leer datos
df_recorridos = pd.read_excel(archivo_matriz, sheet_name='Recorridos')

# Columnas
col_met, col_ruta, col_secuencia = '🏠 MET', '🛣️ Ruta', '🔢 Secuencia'
col_tipo, col_nombre = 'Tipo', '📚 Nombre'
col_lat, col_lon = '🌍 Latitud', '🌍 Longitud'

mets = sorted(df_recorridos[col_met].dropna().unique())
rutas = sorted(df_recorridos[col_ruta].dropna().unique())

# Widgets
met_selector = widgets.SelectMultiple(options=mets, value=tuple(mets), description='METs:', layout=widgets.Layout(width='45%'))
ruta_selector = widgets.SelectMultiple(options=rutas, value=tuple(rutas), description='Rutas:', layout=widgets.Layout(width='45%'))
generar_btn = widgets.Button(description='Generar Mapa Optimizado', button_style='success', layout=widgets.Layout(width='300px'))
output_map = widgets.Output()

def generar_mapa(b):
    with output_map:
        clear_output()
        mets_sel, rutas_sel = list(met_selector.value), list(ruta_selector.value)
        if not mets_sel or not rutas_sel: return
        
        df_filtrado = df_recorridos[(df_recorridos[col_met].isin(mets_sel)) & (df_recorridos[col_ruta].isin(rutas_sel))]
        mapa = folium.Map(location=[df_filtrado[col_lat].mean(), df_filtrado[col_lon].mean()], zoom_start=10)
        colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'black']
        color_idx = 0

        for met in mets_sel:
            met_clean = str(met).replace('MET ', '').strip()
            df_met = df_filtrado[df_filtrado[col_met] == met]
            for ruta in df_met[col_ruta].unique():
                capa_nombre = f"MET {met_clean} - Ruta {ruta}"
                fg = folium.FeatureGroup(name=capa_nombre)
                df_ruta = df_met[df_met[col_ruta] == ruta].sort_values(col_secuencia)
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                
                folium.PolyLine(list(zip(df_ruta[col_lat], df_ruta[col_longitud])), color=color_ruta, weight=5, opacity=0.7).add_to(fg)
                for _, row in df_ruta.iterrows():
                    if row[col_tipo] == 'MET':
                        icon = CustomIcon(icon_met_path, icon_size=(32,32)) if os.path.exists(icon_met_path) else folium.Icon(color='black')
                    else:
                        icon = DivIcon(html=f'<div style="font-size:11px; color:white; background:{color_ruta}; border-radius:50%; width:18px; height:18px; text-align:center; line-height:18px; border:1px solid #fff;">{row[col_secuencia]}</div>')
                    folium.Marker([row[col_lat], row[col_lon]], icon=icon, popup=row[col_nombre]).add_to(fg)
                fg.add_to(mapa)

        # TÍTULO ESTÉTICO
        mapa.get_root().html.add_child(folium.Element(f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 2px solid #fff; min-width: 400px; font-family: sans-serif;">
            <h2 style="margin: 0; text-align: center; font-size:16px;">🗺️ ANÁLISIS COMERCIAL</h2>
            <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px;">
                📍 METs: <b>{len(mets_sel)}</b> | 🛣️ Rutas: <b>{len(rutas_sel)}</b> | 👥 Clientes: <b>{df_filtrado[df_filtrado[col_tipo]=='Cliente'].shape[0]}</b>
            </p>
        </div>
        '''))

        # CSS PARA BOTONES OPTIMIZADOS
        mapa.get_root().html.add_child(folium.Element('''
        <style>
        .leaflet-control-layers-overlays { display: none !important; }
        .leaflet-control-layers-list { 
            max-height: 400px; /* Limita la altura total */
            overflow-y: auto; /* Activa la barra de desplazamiento */
            min-width: 260px; 
            font-family: 'Segoe UI', sans-serif; 
        }
        .layer-section-title { 
            font-size: 10px; font-weight: bold; color: #444; padding: 6px 10px; 
            text-transform: uppercase; background: #f0f0f0; position: sticky; top: 0; z-index: 10;
        }
        .layer-control-buttons, .met-buttons-row { 
            padding: 8px; display: flex; gap: 4px; flex-wrap: wrap; border-bottom: 1px solid #eee; 
        }
        /* DISEÑO DE 4 COLUMNAS PARA RUTAS */
        .ruta-buttons-row { 
            padding: 8px; 
            display: grid; 
            grid-template-columns: repeat(4, 1fr); /* Exactamente 4 por fila */
            gap: 4px; 
            border-bottom: 1px solid #eee; 
        }
        .layer-btn, .met-btn, .ruta-btn { 
            padding: 5px 2px; font-size: 10px; cursor: pointer; border-radius: 4px; 
            border: 1px solid #ccc; background: white; text-align: center; 
        }
        .select-all { background: #4CAF50 !important; color: white; border-color: #45a049; }
        .deselect-all { background: #f44336 !important; color: white; border-color: #d32f2f; }
        .met-btn { border-color: #2196F3; color: #1976D2; font-weight: bold; }
        .ruta-btn { border-color: #ff9800; color: #e65100; font-weight: bold; }
        .met-btn:hover { background: #2196F3; color: white; }
        .ruta-btn:hover { background: #ff9800; color: white; }
        </style>
        '''))

        # JS PARA INYECTAR FILTROS
        layer_control_js = '''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            setTimeout(function() {
                const layersList = document.querySelector('.leaflet-control-layers-list');
                const overlaysDiv = document.querySelector('.leaflet-control-layers-overlays');
                if (!overlaysDiv) return;
                
                const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                const mets = new Set(); const rutas = new Set();
                
                checkboxes.forEach(cb => {
                    const txt = cb.parentElement.querySelector('span').textContent;
                    const matchMet = txt.match(/MET ([^\\s-]+)/);
                    const matchRuta = txt.match(/Ruta ([^\\s-]+)/);
                    if(matchMet) mets.add(matchMet[1]);
                    if(matchRuta) rutas.add(matchRuta[1]);
                });

                const container = document.createElement('div');
                
                // 1. GESTIÓN GENERAL
                container.innerHTML += '<div class="layer-section-title">General</div><div class="layer-control-buttons"><button class="layer-btn select-all">Todo</button><button class="layer-btn deselect-all">Nada</button></div>';
                
                // 2. FILTRO POR MET
                let metHtml = '<div class="layer-section-title">Por MET</div><div class="met-buttons-row">';
                Array.from(mets).sort().forEach(m => metHtml += `<button class="met-btn" data-met="${m}">${m}</button>`);
                container.innerHTML += metHtml + '</div>';

                // 3. FILTRO POR RUTA (4 por fila mediante CSS)
                let rutaHtml = '<div class="layer-section-title">Por N° Ruta (Multi-MET)</div><div class="ruta-buttons-row">';
                Array.from(rutas).sort((a,b) => a-b).forEach(r => rutaHtml += `<button class="ruta-btn" data-ruta="${r}">R${r}</button>`);
                container.innerHTML += rutaHtml + '</div>';

                layersList.insertBefore(container, overlaysDiv);

                container.addEventListener('click', (e) => {
                    if(!e.target.classList.contains('layer-btn') && !e.target.classList.contains('met-btn') && !e.target.classList.contains('ruta-btn')) return;
                    
                    const isAll = e.target.classList.contains('select-all');
                    const isNone = e.target.classList.contains('deselect-all');
                    const targetMet = e.target.getAttribute('data-met');
                    const targetRuta = e.target.getAttribute('data-ruta');

                    checkboxes.forEach(cb => {
                        const txt = cb.parentElement.querySelector('span').textContent;
                        if(isAll) { if(!cb.checked) cb.click(); }
                        else if(isNone) { if(cb.checked) cb.click(); }
                        else if(targetMet) {
                            const match = txt.includes('MET ' + targetMet);
                            if(match && !cb.checked) cb.click(); else if(!match && cb.checked) cb.click();
                        }
                        else if(targetRuta) {
                            const match = txt.endsWith('Ruta ' + targetRuta);
                            if(match && !cb.checked) cb.click(); else if(!match && cb.checked) cb.click();
                        }
                    });
                });
            }, 1500);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(layer_control_js))
        folium.LayerControl(collapsed=False).add_to(mapa)
        
        path_html = os.path.join(output_dir, f'mapa_final_{datetime.now().strftime("%H%M%S")}.html')
        mapa.save(path_html)
        print(f'✅ Guardado en: {path_html}')

generar_btn.on_click(generar_mapa)
display(widgets.VBox([met_selector, ruta_selector, generar_btn, output_map]))

In [ ]:
# ================== MAPA CON POPUPS ESTÉTICOS Y DATOS DETALLADOS ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuración de rutas
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Inputs_mapa'
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
os.makedirs(output_dir, exist_ok=True)

archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))
archivo_matriz = archivos_excel[0] if archivos_excel else None

# Leer datos
df_recorridos = pd.read_excel(archivo_matriz, sheet_name='Recorridos')

# Columnas
col_met, col_ruta, col_secuencia = '🏠 MET', '🛣️ Ruta', '🔢 Secuencia'
col_tipo, col_nombre, col_codigo = 'Tipo', '📚 Nombre', '🆔 Codigo'
col_lat, col_lon = '🌍 Latitud', '🌍 Longitud'

mets = sorted(df_recorridos[col_met].dropna().unique())
rutas = sorted(df_recorridos[col_ruta].dropna().unique())

# Widgets
met_selector = widgets.SelectMultiple(options=mets, value=tuple(mets), description='METs:', layout=widgets.Layout(width='45%'))
ruta_selector = widgets.SelectMultiple(options=rutas, value=tuple(rutas), description='Rutas:', layout=widgets.Layout(width='45%'))
generar_btn = widgets.Button(description='Generar Mapa Estético', button_style='success', layout=widgets.Layout(width='300px'))
output_map = widgets.Output()

def generar_mapa(b):
    with output_map:
        clear_output()
        mets_sel, rutas_sel = list(met_selector.value), list(ruta_selector.value)
        if not mets_sel or not rutas_sel: return
        
        df_filtrado = df_recorridos[(df_recorridos[col_met].isin(mets_sel)) & (df_recorridos[col_ruta].isin(rutas_sel))]
        mapa = folium.Map(location=[df_filtrado[col_lat].mean(), df_filtrado[col_lon].mean()], zoom_start=10)
        colores = ['#E63946', '#457B9D', '#2A9D8F', '#8338EC', '#F4A261', '#7209B7', '#0077B6', '#000000']
        color_idx = 0

        for met in mets_sel:
            met_clean = str(met).replace('MET ', '').strip()
            df_met = df_filtrado[df_filtrado[col_met] == met]
            for ruta in df_met[col_ruta].unique():
                capa_nombre = f"MET {met_clean} - Ruta {ruta}"
                fg = folium.FeatureGroup(name=capa_nombre)
                df_ruta = df_met[df_met[col_ruta] == ruta].sort_values(col_secuencia)
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                
                folium.PolyLine(list(zip(df_ruta[col_lat], df_ruta[col_lon])), color=color_ruta, weight=5, opacity=0.6).add_to(fg)
                
                for _, row in df_ruta.iterrows():
                    # --- DISEÑO DEL POPUP ESTÉTICO ---
                    tipo_color = "#333" if row[col_tipo] == 'Cliente' else "#E63946"
                    popup_html = f'''
                    <div style="font-family: 'Segoe UI', sans-serif; width: 220px; padding: 10px; border-radius: 10px; background-color: #fff; box-shadow: 0 2px 5px rgba(0,0,0,0.2);">
                        <div style="font-size: 10px; color: #888; text-transform: uppercase; font-weight: bold; margin-bottom: 3px;">{row[col_tipo]}</div>
                        <div style="font-size: 14px; font-weight: bold; color: {tipo_color}; margin-bottom: 5px;">ID: {row[col_codigo]}</div>
                        <div style="font-size: 13px; color: #444; border-top: 1px solid #eee; padding-top: 5px; line-height: 1.3;"><b>Nombre:</b><br>{row[col_nombre]}</div>
                        <div style="margin-top: 8px; font-size: 11px; background-color: {color_ruta}22; color: {color_ruta}; padding: 4px; border-radius: 5px; text-align: center;">
                            Ruta {row[col_ruta]} | Secuencia {row[col_secuencia]}
                        </div>
                    </div>
                    '''
                    
                    if row[col_tipo] == 'MET':
                        icon = CustomIcon(icon_met_path, icon_size=(34,34)) if os.path.exists(icon_met_path) else folium.Icon(color='red', icon='home')
                    else:
                        icon = DivIcon(html=f'''
                            <div style="
                                font-size: 10px; 
                                color: white; 
                                background: {color_ruta}; 
                                border-radius: 50%; 
                                width: 22px; 
                                height: 22px; 
                                text-align: center; 
                                line-height: 22px; 
                                border: 2px solid #fff; 
                                font-weight: bold;
                                box-shadow: 0 2px 4px rgba(0,0,0,0.3);
                            ">{row[col_secuencia]}</div>''')

                    folium.Marker(
                        [row[col_lat], row[col_lon]], 
                        icon=icon, 
                        popup=folium.Popup(popup_html, max_width=250),
                        tooltip=f"[{row[col_codigo]}] {row[col_nombre]}" # Etiqueta al pasar el ratón
                    ).add_to(fg)
                fg.add_to(mapa)

        # TÍTULO ESTÉTICO DINÁMICO
        mapa.get_root().html.add_child(folium.Element(f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 1px solid rgba(255,255,255,0.3); min-width: 400px; font-family: 'Segoe UI', sans-serif;">
            <h2 style="margin: 0; text-align: center; font-size:16px; letter-spacing: 1px;">🗺️ LOGÍSTICA DE RECORRIDOS</h2>
            <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.9;">
                📍 METs: <b>{len(mets_sel)}</b> | 🛣️ Rutas: <b>{len(rutas_sel)}</b> | 👥 Clientes: <b>{df_filtrado[df_filtrado[col_tipo]=='Cliente'].shape[0]}</b>
            </p>
        </div>
        '''))

        # CSS PARA BOTONES OPTIMIZADOS
        mapa.get_root().html.add_child(folium.Element('''
        <style>
        .leaflet-control-layers-overlays { display: none !important; }
        .leaflet-control-layers-list { max-height: 400px; overflow-y: auto; min-width: 260px; font-family: 'Segoe UI', sans-serif; border-radius: 8px; }
        .layer-section-title { font-size: 10px; font-weight: bold; color: #555; padding: 8px 10px; text-transform: uppercase; background: #f8f9fa; position: sticky; top: 0; z-index: 10; border-bottom: 1px solid #ddd; }
        .layer-control-buttons, .met-buttons-row { padding: 8px; display: flex; gap: 4px; flex-wrap: wrap; border-bottom: 1px solid #eee; }
        .ruta-buttons-row { padding: 8px; display: grid; grid-template-columns: repeat(4, 1fr); gap: 4px; border-bottom: 1px solid #eee; }
        .layer-btn, .met-btn, .ruta-btn { padding: 5px 2px; font-size: 10px; cursor: pointer; border-radius: 4px; border: 1px solid #ddd; background: white; text-align: center; transition: 0.2s; }
        .select-all { background: #2a9d8f !important; color: white; border: none; }
        .deselect-all { background: #e63946 !important; color: white; border: none; }
        .met-btn { border-color: #457b9d; color: #1d3557; font-weight: 600; }
        .ruta-btn { border-color: #f4a261; color: #e76f51; font-weight: 600; }
        .met-btn:hover { background: #457b9d; color: white; }
        .ruta-btn:hover { background: #f4a261; color: white; }
        </style>
        '''))

        # JS PARA INYECTAR FILTROS
        layer_control_js = '''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            setTimeout(function() {
                const layersList = document.querySelector('.leaflet-control-layers-list');
                const overlaysDiv = document.querySelector('.leaflet-control-layers-overlays');
                if (!overlaysDiv) return;
                const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                const mets = new Set(); const rutas = new Set();
                checkboxes.forEach(cb => {
                    const txt = cb.parentElement.querySelector('span').textContent;
                    const mMet = txt.match(/MET ([^\\s-]+)/);
                    const mRuta = txt.match(/Ruta ([^\\s-]+)/);
                    if(mMet) mets.add(mMet[1]);
                    if(mRuta) rutas.add(mRuta[1]);
                });
                const container = document.createElement('div');
                container.innerHTML += '<div class="layer-section-title">General</div><div class="layer-control-buttons"><button class="layer-btn select-all">Todo</button><button class="layer-btn deselect-all">Nada</button></div>';
                let metHtml = '<div class="layer-section-title">Por MET</div><div class="met-buttons-row">';
                Array.from(mets).sort().forEach(m => metHtml += `<button class="met-btn" data-met="${m}">${m}</button>`);
                container.innerHTML += metHtml + '</div>';
                let rutaHtml = '<div class="layer-section-title">Por N° Ruta</div><div class="ruta-buttons-row">';
                Array.from(rutas).sort((a,b) => a-b).forEach(r => rutaHtml += `<button class="ruta-btn" data-ruta="${r}">R${r}</button>`);
                container.innerHTML += rutaHtml + '</div>';
                layersList.insertBefore(container, overlaysDiv);
                container.addEventListener('click', (e) => {
                    if(!e.target.tagName === 'BUTTON') return;
                    const isAll = e.target.classList.contains('select-all');
                    const isNone = e.target.classList.contains('deselect-all');
                    const tMet = e.target.getAttribute('data-met');
                    const tRuta = e.target.getAttribute('data-ruta');
                    checkboxes.forEach(cb => {
                        const txt = cb.parentElement.querySelector('span').textContent;
                        if(isAll) { if(!cb.checked) cb.click(); }
                        else if(isNone) { if(cb.checked) cb.click(); }
                        else if(tMet) {
                            const match = txt.includes('MET ' + tMet);
                            if(match && !cb.checked) cb.click(); else if(!match && cb.checked) cb.click();
                        }
                        else if(tRuta) {
                            const match = txt.endsWith('Ruta ' + tRuta);
                            if(match && !cb.checked) cb.click(); else if(!match && cb.checked) cb.click();
                        }
                    });
                });
            }, 1500);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(layer_control_js))
        folium.LayerControl(collapsed=False).add_to(mapa)
        
        path_html = os.path.join(output_dir, f'mapa_clientes_{datetime.now().strftime("%H%M%S")}.html')
        mapa.save(path_html)
        print(f'✅ Mapa Estético Guardado: {path_html}')

generar_btn.on_click(generar_mapa)
display(widgets.VBox([met_selector, ruta_selector, generar_btn, output_map]))

In [ ]:
# ================== MAPA CON FILTROS Y ETIQUETAS DE MET DETALLADAS ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuración de rutas
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Inputs_mapa'
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
os.makedirs(output_dir, exist_ok=True)

archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))
archivo_matriz = archivos_excel[0] if archivos_excel else None

# Leer datos
df_recorridos = pd.read_excel(archivo_matriz, sheet_name='Recorridos')

# Columnas
col_met, col_ruta, col_secuencia = '🏠 MET', '🛣️ Ruta', '🔢 Secuencia'
col_tipo, col_nombre, col_codigo = 'Tipo', '📚 Nombre', '🆔 Codigo'
col_lat, col_lon = '🌍 Latitud', '🌍 Longitud'

mets = sorted(df_recorridos[col_met].dropna().unique())
rutas = sorted(df_recorridos[col_ruta].dropna().unique())

# Widgets
met_selector = widgets.SelectMultiple(options=mets, value=tuple(mets), description='METs:', layout=widgets.Layout(width='45%'))
ruta_selector = widgets.SelectMultiple(options=rutas, value=tuple(rutas), description='Rutas:', layout=widgets.Layout(width='45%'))
generar_btn = widgets.Button(description='Generar Mapa Estético', button_style='success', layout=widgets.Layout(width='300px'))
output_map = widgets.Output()

def generar_mapa(b):
    with output_map:
        clear_output()
        mets_sel, rutas_sel = list(met_selector.value), list(ruta_selector.value)
        if not mets_sel or not rutas_sel: return
        
        df_filtrado = df_recorridos[(df_recorridos[col_met].isin(mets_sel)) & (df_recorridos[col_ruta].isin(rutas_sel))]
        mapa = folium.Map(location=[df_filtrado[col_lat].mean(), df_filtrado[col_lon].mean()], zoom_start=10)
        colores = ['#E63946', '#457B9D', '#2A9D8F', '#8338EC', '#F4A261', '#7209B7', '#0077B6', '#000000']
        color_idx = 0

        for met in mets_sel:
            met_clean = str(met).replace('MET ', '').strip()
            df_met = df_filtrado[df_filtrado[col_met] == met]
            for ruta in df_met[col_ruta].unique():
                capa_nombre = f"MET {met_clean} - Ruta {ruta}"
                fg = folium.FeatureGroup(name=capa_nombre)
                df_ruta = df_met[df_met[col_ruta] == ruta].sort_values(col_secuencia)
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                
                folium.PolyLine(list(zip(df_ruta[col_lat], df_ruta[col_lon])), color=color_ruta, weight=5, opacity=0.6).add_to(fg)
                
                for _, row in df_ruta.iterrows():
                    # --- LÓGICA DE POPUP Y TOOLTIP SEGÚN TIPO ---
                    if row[col_tipo] == 'MET':
                        # Popup específico para MET con coordenadas
                        popup_html = f'''
                        <div style="font-family: 'Segoe UI', sans-serif; width: 220px; padding: 10px; border-radius: 10px; background-color: #f0f4f8; border: 2px solid #1d3557;">
                            <div style="font-size: 10px; color: #1d3557; text-transform: uppercase; font-weight: bold; margin-bottom: 3px;">🏠 PUNTO DE ORIGEN</div>
                            <div style="font-size: 14px; font-weight: bold; color: #1d3557; margin-bottom: 5px;">{row[col_met]}</div>
                            <div style="font-size: 13px; color: #444; border-top: 1px solid #1d355744; padding-top: 5px; line-height: 1.3;">
                                <b>Coordenadas:</b><br>
                                <span style="font-family: monospace; color: #e63946;">{row[col_lat]:.6f}, {row[col_lon]:.6f}</span>
                            </div>
                        </div>
                        '''
                        tooltip_txt = f"🏠 MET: {row[col_met]} | {row[col_lat]:.6f}, {row[col_lon]:.6f}"
                        icon = CustomIcon(icon_met_path, icon_size=(34,34)) if os.path.exists(icon_met_path) else folium.Icon(color='red', icon='home')
                    else:
                        # Popup estándar para Clientes
                        popup_html = f'''
                        <div style="font-family: 'Segoe UI', sans-serif; width: 220px; padding: 10px; border-radius: 10px; background-color: #fff; box-shadow: 0 2px 5px rgba(0,0,0,0.2);">
                            <div style="font-size: 10px; color: #888; text-transform: uppercase; font-weight: bold; margin-bottom: 3px;">{row[col_tipo]}</div>
                            <div style="font-size: 14px; font-weight: bold; color: #333; margin-bottom: 5px;">ID: {row[col_codigo]}</div>
                            <div style="font-size: 13px; color: #444; border-top: 1px solid #eee; padding-top: 5px; line-height: 1.3;"><b>Nombre:</b><br>{row[col_nombre]}</div>
                            <div style="margin-top: 8px; font-size: 11px; background-color: {color_ruta}22; color: {color_ruta}; padding: 4px; border-radius: 5px; text-align: center;">
                                Ruta {row[col_ruta]} | Secuencia {row[col_secuencia]}
                            </div>
                        </div>
                        '''
                        tooltip_txt = f"[{row[col_codigo]}] {row[col_nombre]}"
                        icon = DivIcon(html=f'''
                            <div style="
                                font-size: 10px; 
                                color: white; 
                                background: {color_ruta}; 
                                border-radius: 50%; 
                                width: 22px; 
                                height: 22px; 
                                text-align: center; 
                                line-height: 22px; 
                                border: 2px solid #fff; 
                                font-weight: bold;
                                box-shadow: 0 2px 4px rgba(0,0,0,0.3);
                            ">{row[col_secuencia]}</div>''')

                    folium.Marker(
                        [row[col_lat], row[col_lon]], 
                        icon=icon, 
                        popup=folium.Popup(popup_html, max_width=250),
                        tooltip=tooltip_txt
                    ).add_to(fg)
                fg.add_to(mapa)

        # TÍTULO ESTÉTICO
        mapa.get_root().html.add_child(folium.Element(f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 1px solid rgba(255,255,255,0.3); min-width: 400px; font-family: 'Segoe UI', sans-serif;">
            <h2 style="margin: 0; text-align: center; font-size:16px; letter-spacing: 1px;">🗺️ LOGÍSTICA DE RECORRIDOS</h2>
            <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.9;">
                📍 METs: <b>{len(mets_sel)}</b> | 🛣️ Rutas: <b>{len(rutas_sel)}</b> | 👥 Clientes: <b>{df_filtrado[df_filtrado[col_tipo]=='Cliente'].shape[0]}</b>
            </p>
        </div>
        '''))

        # CSS PARA FILTROS (GRID 4 COLUMNAS + SCROLL)
        mapa.get_root().html.add_child(folium.Element('''
        <style>
        .leaflet-control-layers-overlays { display: none !important; }
        .leaflet-control-layers-list { max-height: 400px; overflow-y: auto; min-width: 260px; font-family: 'Segoe UI', sans-serif; border-radius: 8px; }
        .layer-section-title { font-size: 10px; font-weight: bold; color: #555; padding: 8px 10px; text-transform: uppercase; background: #f8f9fa; position: sticky; top: 0; z-index: 10; border-bottom: 1px solid #ddd; }
        .layer-control-buttons, .met-buttons-row { padding: 8px; display: flex; gap: 4px; flex-wrap: wrap; border-bottom: 1px solid #eee; }
        .ruta-buttons-row { padding: 8px; display: grid; grid-template-columns: repeat(4, 1fr); gap: 4px; border-bottom: 1px solid #eee; }
        .layer-btn, .met-btn, .ruta-btn { padding: 5px 2px; font-size: 10px; cursor: pointer; border-radius: 4px; border: 1px solid #ddd; background: white; text-align: center; transition: 0.2s; }
        .select-all { background: #2a9d8f !important; color: white; border: none; }
        .deselect-all { background: #e63946 !important; color: white; border: none; }
        .met-btn { border-color: #457b9d; color: #1d3557; font-weight: 600; }
        .ruta-btn { border-color: #f4a261; color: #e76f51; font-weight: 600; }
        .met-btn:hover { background: #457b9d; color: white; }
        .ruta-btn:hover { background: #f4a261; color: white; }
        </style>
        '''))

        # JS PARA FILTROS
        layer_control_js = '''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            setTimeout(function() {
                const layersList = document.querySelector('.leaflet-control-layers-list');
                const overlaysDiv = document.querySelector('.leaflet-control-layers-overlays');
                if (!overlaysDiv) return;
                const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                const mets = new Set(); const rutas = new Set();
                checkboxes.forEach(cb => {
                    const txt = cb.parentElement.querySelector('span').textContent;
                    const mMet = txt.match(/MET ([^\\s-]+)/);
                    const mRuta = txt.match(/Ruta ([^\\s-]+)/);
                    if(mMet) mets.add(mMet[1]);
                    if(mRuta) rutas.add(mRuta[1]);
                });
                const container = document.createElement('div');
                container.innerHTML += '<div class="layer-section-title">General</div><div class="layer-control-buttons"><button class="layer-btn select-all">Todo</button><button class="layer-btn deselect-all">Nada</button></div>';
                let metHtml = '<div class="layer-section-title">Por MET</div><div class="met-buttons-row">';
                Array.from(mets).sort().forEach(m => metHtml += `<button class="met-btn" data-met="${m}">${m}</button>`);
                container.innerHTML += metHtml + '</div>';
                let rutaHtml = '<div class="layer-section-title">Por N° Ruta</div><div class="ruta-buttons-row">';
                Array.from(rutas).sort((a,b) => a-b).forEach(r => rutaHtml += `<button class="ruta-btn" data-ruta="${r}">R${r}</button>`);
                container.innerHTML += rutaHtml + '</div>';
                layersList.insertBefore(container, overlaysDiv);
                container.addEventListener('click', (e) => {
                    if(!e.target.tagName === 'BUTTON') return;
                    const isAll = e.target.classList.contains('select-all');
                    const isNone = e.target.classList.contains('deselect-all');
                    const tMet = e.target.getAttribute('data-met');
                    const tRuta = e.target.getAttribute('data-ruta');
                    checkboxes.forEach(cb => {
                        const txt = cb.parentElement.querySelector('span').textContent;
                        if(isAll) { if(!cb.checked) cb.click(); }
                        else if(isNone) { if(cb.checked) cb.click(); }
                        else if(tMet) {
                            const match = txt.includes('MET ' + tMet);
                            if(match && !cb.checked) cb.click(); else if(!match && cb.checked) cb.click();
                        }
                        else if(tRuta) {
                            const match = txt.endsWith('Ruta ' + tRuta);
                            if(match && !cb.checked) cb.click(); else if(!match && cb.checked) cb.click();
                        }
                    });
                });
            }, 1500);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(layer_control_js))
        folium.LayerControl(collapsed=False).add_to(mapa)
        
        path_html = os.path.join(output_dir, f'mapa_final_{datetime.now().strftime("%H%M%S")}.html')
        mapa.save(path_html)
        print(f'✅ Mapa Generado con Éxito: {path_html}')

generar_btn.on_click(generar_mapa)
display(widgets.VBox([met_selector, ruta_selector, generar_btn, output_map]))

<>:142: SyntaxWarning: invalid escape sequence '\s'
<>:142: SyntaxWarning: invalid escape sequence '\s'
C:\Users\igcamposg\AppData\Local\Temp\ipykernel_27324\1994708007.py:142: SyntaxWarning: invalid escape sequence '\s'
  const m = txt.match(/MET\s+([^\\s-]+)/);


In [ ]:
# ================== MAPA CON FILTROS INTELIGENTES Y DISEÑO ESTÉTICO ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Configuración de rutas
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Inputs_mapa'
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
os.makedirs(output_dir, exist_ok=True)

archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))
archivo_matriz = archivos_excel[0] if archivos_excel else None

# 2. Leer datos
df_recorridos = pd.read_excel(archivo_matriz, sheet_name='Recorridos')

# Columnas
col_met, col_ruta, col_secuencia = '🏠 MET', '🛣️ Ruta', '🔢 Secuencia'
col_tipo, col_nombre, col_codigo = 'Tipo', '📚 Nombre', '🆔 Codigo'
col_lat, col_lon = '🌍 Latitud', '🌍 Longitud'

mets = sorted(df_recorridos[col_met].dropna().unique())
rutas = sorted(df_recorridos[col_ruta].dropna().unique())

# Widgets de interfaz
met_selector = widgets.SelectMultiple(options=mets, value=tuple(mets), description='METs:', layout=widgets.Layout(width='45%'))
ruta_selector = widgets.SelectMultiple(options=rutas, value=tuple(rutas), description='Rutas:', layout=widgets.Layout(width='45%'))
generar_btn = widgets.Button(description='Generar Mapa Final', button_style='success', layout=widgets.Layout(width='300px'))
output_map = widgets.Output()

def generar_mapa(b):
    with output_map:
        clear_output()
        mets_sel, rutas_sel = list(met_selector.value), list(ruta_selector.value)
        if not mets_sel or not rutas_sel: return
        
        df_filtrado = df_recorridos[(df_recorridos[col_met].isin(mets_sel)) & (df_recorridos[col_ruta].isin(rutas_sel))]
        mapa = folium.Map(location=[df_filtrado[col_lat].mean(), df_filtrado[col_lon].mean()], zoom_start=10)
        
        # Paleta de colores estética
        colores = ['#E63946', '#457B9D', '#2A9D8F', '#8338EC', '#F4A261', '#7209B7', '#0077B6', '#264653']
        color_idx = 0

        # 3. Creación de Capas
        for met in mets_sel:
            met_clean = str(met).replace('MET ', '').strip()
            df_met = df_filtrado[df_filtrado[col_met] == met]
            for ruta in sorted(df_met[col_ruta].unique()):
                capa_nombre = f"MET {met_clean} - Ruta {ruta}"
                fg = folium.FeatureGroup(name=capa_nombre)
                
                df_ruta = df_met[df_met[col_ruta] == ruta].sort_values(col_secuencia)
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                
                # Dibujar Trazado
                folium.PolyLine(list(zip(df_ruta[col_lat], df_ruta[col_lon])), color=color_ruta, weight=5, opacity=0.6).add_to(fg)
                
                for _, row in df_ruta.iterrows():
                    if row[col_tipo] == 'MET':
                        popup_html = f'''
                        <div style="font-family: 'Segoe UI', sans-serif; width: 200px; padding: 5px;">
                            <b style="color: #1d3557;">🏠 ORIGEN: {row[col_met]}</b><br>
                            <hr style="margin: 5px 0;">
                            <small>Lat: {row[col_lat]:.5f}<br>Lon: {row[col_lon]:.5f}</small>
                        </div>
                        '''
                        icon = CustomIcon(icon_met_path, icon_size=(34,34)) if os.path.exists(icon_met_path) else folium.Icon(color='red', icon='home')
                    else:
                        popup_html = f'''
                        <div style="font-family: 'Segoe UI', sans-serif; width: 200px;">
                            <div style="font-size: 11px; font-weight: bold; color: {color_ruta};">RUTA {row[col_ruta]} | POS {row[col_secuencia]}</div>
                            <b style="font-size: 13px;">ID: {row[col_codigo]}</b><br>
                            <span style="font-size: 12px; color: #555;">{row[col_nombre]}</span>
                        </div>
                        '''
                        icon = DivIcon(html=f'''
                            <div style="font-size: 10px; color: white; background: {color_ruta}; border-radius: 50%; width: 22px; height: 22px; text-align: center; line-height: 22px; border: 2px solid #fff; font-weight: bold; box-shadow: 0 2px 4px rgba(0,0,0,0.3);">
                            {row[col_secuencia]}</div>''')

                    folium.Marker([row[col_lat], row[col_lon]], icon=icon, popup=folium.Popup(popup_html, max_width=250)).add_to(fg)
                fg.add_to(mapa)

        # 4. TÍTULO FLOTANTE
        mapa.get_root().html.add_child(folium.Element(f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9999; background: #1d3557; color: white; padding: 10px 20px; border-radius: 12px; font-family: 'Segoe UI', sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.4); text-align: center; min-width: 300px;">
            <span style="font-size: 15px; font-weight: bold; letter-spacing: 1px;">🗺️ VISUALIZADOR LOGÍSTICO</span><br>
            <span style="font-size: 10px; opacity: 0.8;">METs: {len(mets_sel)} | Rutas: {len(rutas_sel)} | Registros: {len(df_filtrado)}</span>
        </div>
        '''))

        # 5. CSS PARA EL MENÚ DE FILTROS
        mapa.get_root().html.add_child(folium.Element('''
        <style>
            .leaflet-control-layers-overlays { display: none !important; }
            .leaflet-control-layers-list { max-height: 420px; overflow-y: auto; min-width: 280px; font-family: 'Segoe UI', sans-serif; padding: 10px; border-radius: 8px; }
            .section-label { font-size: 10px; font-weight: bold; color: #1d3557; padding: 8px 2px 4px; text-transform: uppercase; border-top: 1px solid #eee; margin-top: 5px; }
            .grid-btns { display: grid; grid-template-columns: repeat(4, 1fr); gap: 4px; padding: 5px 0; }
            .flex-btns { display: flex; gap: 4px; padding: 5px 0; flex-wrap: wrap; }
            .btn-logic { padding: 6px 2px; font-size: 10px; cursor: pointer; border-radius: 4px; border: 1px solid #ddd; background: white; font-weight: bold; transition: 0.2s; text-align: center; }
            .btn-logic:hover { background: #f0f7ff; border-color: #457b9d; }
            .all-btn { background: #2a9d8f !important; color: white; border: none; flex: 1; }
            .none-btn { background: #e63946 !important; color: white; border: none; flex: 1; }
            .m-style { border-color: #457b9d; color: #1d3557; width: calc(50% - 4px); }
            .r-style { border-color: #f4a261; color: #e76f51; }
        </style>
        '''))

        # 6. JAVASCRIPT CORREGIDO (Raw String para evitar SyntaxWarning)
        layer_control_js = r'''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            setTimeout(function() {
                const overlays = document.querySelector('.leaflet-control-layers-overlays');
                const list = document.querySelector('.leaflet-control-layers-list');
                if (!overlays || !list) return;
                
                const cbs = overlays.querySelectorAll('input[type="checkbox"]');
                const mets = new Set(); const rutas = new Set();
                
                cbs.forEach(cb => {
                    const txt = cb.parentElement.querySelector('span').textContent;
                    const mMatch = txt.match(/MET\s+([^\s-]+)/);
                    const rMatch = txt.match(/Ruta\s+([^\s-]+)/);
                    if(mMatch) mets.add(mMatch[1]);
                    if(rMatch) rutas.add(rMatch[1]);
                });

                const panel = document.createElement('div');
                
                // General
                panel.innerHTML += '<div class="section-label">General</div><div class="flex-btns"><button class="btn-logic all-btn" id="f-all">Todo</button><button class="btn-logic none-btn" id="f-none">Nada</button></div>';
                
                // METs
                let metH = '<div class="section-label">Por MET (Completo)</div><div class="flex-btns">';
                Array.from(mets).sort().forEach(m => metH += `<button class="btn-logic m-style" data-met="${m}">${m}</button>`);
                panel.innerHTML += metH + '</div>';

                // Rutas
                let rutH = '<div class="section-label">Por N° de Ruta (Multi-MET)</div><div class="grid-btns">';
                Array.from(rutas).sort((a,b) => a-b).forEach(r => rutH += `<button class="btn-logic r-style" data-ruta="${r}">R${r}</button>`);
                panel.innerHTML += rutH + '</div>';

                list.insertBefore(panel, overlays);

                panel.addEventListener('click', (e) => {
                    const target = e.target;
                    if(!target.classList.contains('btn-logic')) return;
                    
                    const tMet = target.getAttribute('data-met');
                    const tRut = target.getAttribute('data-ruta');

                    cbs.forEach(cb => {
                        const label = cb.parentElement.querySelector('span').textContent;
                        if(target.id === 'f-all') { if(!cb.checked) cb.click(); }
                        else if(target.id === 'f-none') { if(cb.checked) cb.click(); }
                        else if(tMet) {
                            const match = label.includes('MET ' + tMet);
                            if(match && !cb.checked) cb.click(); 
                            else if(!match && cb.checked) cb.click();
                        }
                        else if(tRut) {
                            const match = label.endsWith('Ruta ' + tRut);
                            if(match && !cb.checked) cb.click(); 
                            else if(!match && cb.checked) cb.click();
                        }
                    });
                });
            }, 1000);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(layer_control_js))
        folium.LayerControl(collapsed=False).add_to(mapa)
        
        # Guardado
        timestamp = datetime.now().strftime("%H%M%S")
        path_html = os.path.join(output_dir, f'mapa_logistica_{timestamp}.html')
        mapa.save(path_html)
        print(f'✅ Mapa listo en: {path_html}')

generar_btn.on_click(generar_mapa)
display(widgets.VBox([
    widgets.HTML("<h2 style='font-family:sans-serif;'>Configurador de Mapa</h2>"),
    widgets.HBox([met_selector, ruta_selector]),
    generar_btn, 
    output_map
]))

In [ ]:
# ================== MAPA OPTIMIZADO: CARGA LIGERA E INTERFAZ CLONADA ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Configuración de rutas
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Inputs_mapa'
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
os.makedirs(output_dir, exist_ok=True)

archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))
archivo_matriz = archivos_excel[0] if archivos_excel else None

# 2. Leer datos
df_recorridos = pd.read_excel(archivo_matriz, sheet_name='Recorridos')

# Columnas
col_met, col_ruta, col_secuencia = '🏠 MET', '🛣️ Ruta', '🔢 Secuencia'
col_tipo, col_nombre, col_codigo = 'Tipo', '📚 Nombre', '🆔 Codigo'
col_lat, col_lon = '🌍 Latitud', '🌍 Longitud'

mets = sorted(df_recorridos[col_met].dropna().unique())
rutas = sorted(df_recorridos[col_ruta].dropna().unique())

# Widgets
met_selector = widgets.SelectMultiple(options=mets, value=tuple(mets), description='METs:')
ruta_selector = widgets.SelectMultiple(options=rutas, value=tuple(rutas), description='Rutas:')
generar_btn = widgets.Button(description='Generar Mapa Optimizado', button_style='success', layout={'width': '300px'})
output_map = widgets.Output()

def generar_mapa(b):
    with output_map:
        clear_output()
        mets_sel, rutas_sel = list(met_selector.value), list(ruta_selector.value)
        if not mets_sel or not rutas_sel: return
        
        df_filtrado = df_recorridos[(df_recorridos[col_met].isin(mets_sel)) & (df_recorridos[col_ruta].isin(rutas_sel))]
        mapa = folium.Map(location=[df_filtrado[col_lat].mean(), df_filtrado[col_lon].mean()], zoom_start=10)
        
        colores = ['#E63946', '#457B9D', '#2A9D8F', '#8338EC', '#F4A261', '#7209B7', '#0077B6', '#1D3557']
        color_idx = 0

        # --- CAPA ESPECIAL: ICONOS MET (SIEMPRE VISIBLES) ---
        fg_mets_fijos = folium.FeatureGroup(name="🏠 Ubicaciones MET", show=True)
        df_solo_mets = df_filtrado[df_filtrado[col_tipo] == 'MET'].drop_duplicates(subset=[col_met])
        
        for _, row in df_solo_mets.iterrows():
            icon_met = CustomIcon(icon_met_path, icon_size=(36,36)) if os.path.exists(icon_met_path) else folium.Icon(color='red', icon='home')
            folium.Marker(
                [row[col_lat], row[col_lon]], 
                icon=icon_met, 
                popup=f"<b>MET: {row[col_met]}</b>",
                tooltip=f"🏠 {row[col_met]}"
            ).add_to(fg_mets_fijos)
        fg_mets_fijos.add_to(mapa)

        # --- CAPAS DINÁMICAS (OCULTAS POR DEFECTO PARA VELOCIDAD) ---
        for met in mets_sel:
            met_clean = str(met).replace('MET ', '').strip()
            df_met = df_filtrado[df_filtrado[col_met] == met]
            for ruta in sorted(df_met[col_ruta].unique()):
                capa_nombre = f"MET {met_clean} - Ruta {ruta}"
                # IMPORTANTE: show=False hace que el mapa abra rápido
                fg = folium.FeatureGroup(name=capa_nombre, show=False) 
                
                df_ruta = df_met[df_met[col_ruta] == ruta].sort_values(col_secuencia)
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                
                # Líneas
                folium.PolyLine(list(zip(df_ruta[col_lat], df_ruta[col_lon])), color=color_ruta, weight=5, opacity=0.6).add_to(fg)
                
                # Puntos Clientes (Solo se renderizan si el usuario activa la capa)
                for _, row in df_ruta.iterrows():
                    if row[col_tipo] != 'MET':
                        icon_cli = DivIcon(html=f'''
                            <div style="font-size: 10px; color: white; background: {color_ruta}; border-radius: 50%; width: 22px; height: 22px; text-align: center; line-height: 22px; border: 2px solid #fff; font-weight: bold; box-shadow: 0 2px 4px rgba(0,0,0,0.3);">
                            {row[col_secuencia]}</div>''')
                        
                        folium.Marker(
                            [row[col_lat], row[col_lon]], 
                            icon=icon_cli, 
                            popup=folium.Popup(f'<b>ID: {row[col_codigo]}</b><br>{row[col_nombre]}', max_width=250),
                            tooltip=f"[{row[col_codigo]}] {row[col_nombre]}"
                        ).add_to(fg)
                fg.add_to(mapa)

        # 4. TÍTULO ESTÉTICO
        mapa.get_root().html.add_child(folium.Element(f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 1px solid rgba(255,255,255,0.3); min-width: 400px; font-family: 'Segoe UI', sans-serif;">
            <h2 style="margin: 0; text-align: center; font-size:16px; letter-spacing: 1px;">🗺️ LOGÍSTICA DE RECORRIDOS</h2>
            <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.9;">
                📍 METs: <b>{len(mets_sel)}</b> | 🛣️ Rutas: <b>{len(rutas_sel)}</b> | 👥 Clientes: {df_filtrado[df_filtrado[col_tipo]=='Cliente'].shape[0]}
            </p>
        </div>
        '''))

        # 5. CSS CLONADO
        mapa.get_root().html.add_child(folium.Element('''
        <style>
            .leaflet-control-layers-list { padding: 10px; min-width: 280px; font-family: 'Segoe UI', sans-serif; }
            .leaflet-control-layers-overlays { display: none !important; } 
            .layer-control-buttons { display: flex; gap: 5px; margin-bottom: 10px; border-bottom: 1px solid #eee; padding-bottom: 10px; }
            .layer-btn { flex: 1; padding: 6px; cursor: pointer; border: none; border-radius: 4px; font-weight: bold; font-size: 12px; }
            .select-all { background-color: #2a9d8f !important; color: white; }
            .deselect-all { background-color: #e63946 !important; color: white; }
            .met-buttons-row { display: flex; flex-wrap: wrap; gap: 4px; margin-bottom: 10px; }
            .met-btn { padding: 5px 10px; font-size: 11px; cursor: pointer; background: white; border: 1px solid #457b9d; color: #1d3557; border-radius: 4px; font-weight: bold; }
            .ruta-buttons-row { display: grid; grid-template-columns: repeat(4, 1fr); gap: 4px; }
            .ruta-btn { padding: 4px; font-size: 10px; cursor: pointer; background: white; border: 1px solid #f4a261; color: #e76f51; border-radius: 4px; font-weight: bold; }
        </style>
        '''))

        # 6. JS DINÁMICO (Raw String)
        layer_control_js = r'''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            setTimeout(function() {
                const list = document.querySelector('.leaflet-control-layers-list');
                const overlays = document.querySelector('.leaflet-control-layers-overlays');
                if (!list || !overlays) return;

                const cbs = Array.from(overlays.querySelectorAll('input[type="checkbox"]'));
                const mets = new Set(); const rutas = new Set();
                
                cbs.forEach(cb => {
                    const txt = cb.parentElement.querySelector('span').textContent;
                    if (txt.includes("🏠 Ubicaciones MET")) return; 
                    const m = txt.match(/MET\s+([^\s-]+)/);
                    const r = txt.match(/Ruta\s+([^\s-]+)/);
                    if(m) mets.add(m[1]);
                    if(r) rutas.add(r[1]);
                });

                const container = document.createElement('div');
                let html = '<div class="layer-control-buttons"><button class="layer-btn select-all">✓ Todo</button><button class="layer-btn deselect-all">✗ Nada</button></div>';
                
                html += '<div class="met-buttons-row">';
                Array.from(mets).sort().forEach(m => { html += `<button class="met-btn">${m}</button>`; });
                html += '</div>';

                html += '<div class="ruta-buttons-row">';
                Array.from(rutas).sort((a,b) => a-b).forEach(r => { html += `<button class="ruta-btn">R${r}</button>`; });
                html += '</div>';

                container.innerHTML = html;
                list.insertBefore(container, overlays);

                container.addEventListener('click', (e) => {
                    const btn = e.target;
                    const txt = btn.textContent;
                    cbs.forEach(cb => {
                        const label = cb.parentElement.querySelector('span').textContent;
                        if (label.includes("🏠 Ubicaciones MET")) return;

                        if(btn.classList.contains('select-all')) { if(!cb.checked) cb.click(); }
                        else if(btn.classList.contains('deselect-all')) { if(cb.checked) cb.click(); }
                        else if(btn.classList.contains('met-btn')) {
                            const isMet = label.includes('MET ' + txt);
                            if(isMet && !cb.checked) cb.click(); else if(!isMet && cb.checked) cb.click();
                        }
                        else if(btn.classList.contains('ruta-btn')) {
                            const rNum = txt.replace('R', '');
                            const isRuta = label.endsWith('Ruta ' + rNum);
                            if(isRuta && !cb.checked) cb.click(); else if(!isRuta && cb.checked) cb.click();
                        }
                    });
                });
            }, 1000);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(layer_control_js))
        folium.LayerControl(collapsed=False).add_to(mapa)
        
        path_html = os.path.join(output_dir, f'mapa_optimizado_{datetime.now().strftime("%H%M%S")}.html')
        mapa.save(path_html)
        print(f'✅ Mapa optimizado (carga rápida) guardado en: {path_html}')

generar_btn.on_click(generar_mapa)
display(widgets.VBox([met_selector, ruta_selector, generar_btn, output_map]))

In [ ]:
# ================== MAPA LOGÍSTICO PROFESIONAL: COMPLETO Y OPTIMIZADO ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Configuración de rutas y archivos
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Inputs_mapa'
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
os.makedirs(output_dir, exist_ok=True)

archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))
archivo_matriz = archivos_excel[0] if archivos_excel else None

# 2. Lectura de datos
df_recorridos = pd.read_excel(archivo_matriz, sheet_name='Recorridos')

# Definición de columnas
col_met, col_ruta, col_secuencia = '🏠 MET', '🛣️ Ruta', '🔢 Secuencia'
col_tipo, col_nombre, col_codigo = 'Tipo', '📚 Nombre', '🆔 Codigo'
col_lat, col_lon = '🌍 Latitud', '🌍 Longitud'

mets = sorted(df_recorridos[col_met].dropna().unique())
rutas = sorted(df_recorridos[col_ruta].dropna().unique())

# 3. Interfaz de Usuario (Widgets)
met_selector = widgets.SelectMultiple(options=mets, value=tuple(mets), description='METs:', layout={'width': '45%'})
ruta_selector = widgets.SelectMultiple(options=rutas, value=tuple(rutas), description='Rutas:', layout={'width': '45%'})
generar_btn = widgets.Button(description='Generar Mapa Premium', button_style='success', layout={'width': '300px'})
output_map = widgets.Output()

def generar_mapa(b):
    with output_map:
        clear_output()
        mets_sel, rutas_sel = list(met_selector.value), list(ruta_selector.value)
        if not mets_sel or not rutas_sel:
            print("❌ Selecciona al menos un MET y una Ruta.")
            return
        
        # Filtrado de datos
        df_filtrado = df_recorridos[(df_recorridos[col_met].isin(mets_sel)) & (df_recorridos[col_ruta].isin(rutas_sel))]
        
        # Cálculo de clientes por ruta (para las etiquetas)
        conteo_clientes = df_filtrado[df_filtrado[col_tipo] != 'MET'].groupby([col_met, col_ruta]).size().to_dict()
        
        # Mapa base
        mapa = folium.Map(location=[df_filtrado[col_lat].mean(), df_filtrado[col_lon].mean()], zoom_start=10)
        colores = ['#E63946', '#457B9D', '#2A9D8F', '#8338EC', '#F4A261', '#7209B7', '#0077B6', '#1D3557']
        color_idx = 0

        # --- CAPA FIJA: ICONOS MET (SIEMPRE VISIBLES) ---
        fg_mets_fijos = folium.FeatureGroup(name="🏠 Ubicaciones MET", show=True)
        df_solo_mets = df_filtrado[df_filtrado[col_tipo] == 'MET'].drop_duplicates(subset=[col_met])
        for _, row in df_solo_mets.iterrows():
            icon_met = CustomIcon(icon_met_path, icon_size=(36,36)) if os.path.exists(icon_met_path) else folium.Icon(color='red', icon='home')
            folium.Marker([row[col_lat], row[col_lon]], icon=icon_met, tooltip=f"🏠 {row[col_met]}").add_to(fg_mets_fijos)
        fg_mets_fijos.add_to(mapa)

        # --- CAPAS DE RUTAS (OCULTAS POR DEFECTO PARA CARGA RÁPIDA) ---
        for met in mets_sel:
            met_clean = str(met).replace('MET ', '').strip()
            df_met = df_filtrado[df_filtrado[col_met] == met]
            for ruta in sorted(df_met[col_ruta].unique()):
                capa_nombre = f"MET {met_clean} - Ruta {ruta}"
                fg = folium.FeatureGroup(name=capa_nombre, show=False)
                
                df_ruta = df_met[df_met[col_ruta] == ruta].sort_values(col_secuencia)
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                total_clientes_ruta = conteo_clientes.get((met, ruta), 0)
                
                # Dibujar Línea de Recorrido
                folium.PolyLine(list(zip(df_ruta[col_lat], df_ruta[col_lon])), color=color_ruta, weight=5, opacity=0.6).add_to(fg)
                
                # Dibujar Marcadores de Clientes
                for _, row in df_ruta.iterrows():
                    if row[col_tipo] != 'MET':
                        # Estética mejorada del Popup
                        popup_html = f'''
                        <div style="font-family: 'Segoe UI', sans-serif; width: 220px; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 10px rgba(0,0,0,0.2); border: 1px solid #ddd;">
                            <div style="background: linear-gradient(135deg, {color_ruta} 0%, #333 100%); color: white; padding: 10px; text-align: center;">
                                <div style="font-size: 10px; opacity: 0.8; text-transform: uppercase;">Ruta {row[col_ruta]}</div>
                                <div style="font-size: 15px; font-weight: bold;">ID: {row[col_codigo]}</div>
                            </div>
                            <div style="padding: 12px; background: white;">
                                <div style="font-size: 13px; font-weight: 600; color: #333; margin-bottom: 5px;">{row[col_nombre]}</div>
                                <div style="display: flex; justify-content: space-between; font-size: 11px; color: #666; border-top: 1px solid #eee; padding-top: 8px;">
                                    <span><b>Posición:</b> {row[col_secuencia]}</span>
                                    <span><b>Paradas:</b> {total_clientes_ruta}</span>
                                </div>
                            </div>
                        </div>
                        '''
                        icon_cli = DivIcon(html=f'''
                            <div style="font-size: 10px; color: white; background: {color_ruta}; border-radius: 50%; width: 24px; height: 24px; text-align: center; line-height: 24px; border: 2px solid #fff; font-weight: bold;">
                            {row[col_secuencia]}</div>''')
                        
                        folium.Marker(
                            [row[col_lat], row[col_lon]], 
                            icon=icon_cli, 
                            popup=folium.Popup(popup_html, max_width=250),
                            tooltip=f"ID: {row[col_codigo]} | R{row[col_ruta]}"
                        ).add_to(fg)
                fg.add_to(mapa)

        # 4. TÍTULO ESTÉTICO FLOTANTE
        mapa.get_root().html.add_child(folium.Element(f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9999; background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 1px solid rgba(255,255,255,0.3); min-width: 400px; font-family: 'Segoe UI', sans-serif;">
            <h2 style="margin: 0; text-align: center; font-size:16px;">🗺️ ANÁLISIS DE RECORRIDOS</h2>
            <p style="margin: 4px 0 0 0; text-align: center; font-size: 11px; opacity: 0.9;">
                📍 METs: {len(mets_sel)} | 🛣️ Rutas: {len(rutas_sel)} | 👥 Clientes: {df_filtrado[df_filtrado[col_tipo]=='Cliente'].shape[0]}
            </p>
        </div>
        '''))

        # 5. CSS CLONADO (Colores y Grids)
        mapa.get_root().html.add_child(folium.Element('''
        <style>
            .leaflet-control-layers-list { padding: 10px; min-width: 280px; font-family: 'Segoe UI', sans-serif; border-radius: 8px; }
            .leaflet-control-layers-overlays { display: none !important; } 
            .layer-control-buttons { display: flex; gap: 5px; margin-bottom: 10px; border-bottom: 1px solid #eee; padding-bottom: 10px; }
            .layer-btn { flex: 1; padding: 6px; cursor: pointer; border: none; border-radius: 4px; font-weight: bold; font-size: 12px; transition: 0.2s; }
            .select-all { background-color: #2a9d8f !important; color: white; }
            .deselect-all { background-color: #e63946 !important; color: white; }
            .met-buttons-row { display: flex; flex-wrap: wrap; gap: 4px; margin-bottom: 10px; }
            .met-btn { padding: 5px 10px; font-size: 11px; cursor: pointer; background: white; border: 1px solid #457b9d; color: #1d3557; border-radius: 4px; font-weight: bold; }
            .met-btn:hover { background: #457b9d; color: white; }
            .ruta-buttons-row { display: grid; grid-template-columns: repeat(4, 1fr); gap: 4px; }
            .ruta-btn { padding: 4px; font-size: 10px; cursor: pointer; background: white; border: 1px solid #f4a261; color: #e76f51; border-radius: 4px; font-weight: bold; }
            .ruta-btn:hover { background: #f4a261; color: white; }
        </style>
        '''))

        # 6. JS DINÁMICO (Raw String corregido)
        layer_control_js = r'''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            setTimeout(function() {
                const list = document.querySelector('.leaflet-control-layers-list');
                const overlays = document.querySelector('.leaflet-control-layers-overlays');
                if (!list || !overlays) return;

                const cbs = Array.from(overlays.querySelectorAll('input[type="checkbox"]'));
                const mets = new Set(); const rutas = new Set();
                
                cbs.forEach(cb => {
                    const txt = cb.parentElement.querySelector('span').textContent;
                    if (txt.includes("🏠 Ubicaciones MET")) return; 
                    const m = txt.match(/MET\s+([^\s-]+)/);
                    const r = txt.match(/Ruta\s+([^\s-]+)/);
                    if(m) mets.add(m[1]);
                    if(r) rutas.add(r[1]);
                });

                const container = document.createElement('div');
                let html = '<div class="layer-control-buttons"><button class="layer-btn select-all">✓ Todo</button><button class="layer-btn deselect-all">✗ Nada</button></div>';
                
                html += '<div class="met-buttons-row">';
                Array.from(mets).sort().forEach(m => { html += `<button class="met-btn" title="Solo MET ${m}">${m}</button>`; });
                html += '</div>';

                html += '<div class="ruta-buttons-row">';
                Array.from(rutas).sort((a,b) => a-b).forEach(r => { html += `<button class="ruta-btn" title="Ruta ${r} todos los MET">R${r}</button>`; });
                html += '</div>';

                container.innerHTML = html;
                list.insertBefore(container, overlays);

                container.addEventListener('click', (e) => {
                    const btn = e.target;
                    const txt = btn.textContent;
                    if(!btn.classList.contains('layer-btn') && !btn.classList.contains('met-btn') && !btn.classList.contains('ruta-btn')) return;

                    cbs.forEach(cb => {
                        const label = cb.parentElement.querySelector('span').textContent;
                        if (label.includes("🏠 Ubicaciones MET")) return;

                        if(btn.classList.contains('select-all')) { if(!cb.checked) cb.click(); }
                        else if(btn.classList.contains('deselect-all')) { if(cb.checked) cb.click(); }
                        else if(btn.classList.contains('met-btn')) {
                            const isMet = label.includes('MET ' + txt);
                            if(isMet && !cb.checked) cb.click(); else if(!isMet && cb.checked) cb.click();
                        }
                        else if(btn.classList.contains('ruta-btn')) {
                            const rNum = txt.replace('R', '');
                            const isRuta = label.endsWith('Ruta ' + rNum);
                            if(isRuta && !cb.checked) cb.click(); else if(!isRuta && cb.checked) cb.click();
                        }
                    });
                });
            }, 1000);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(layer_control_js))
        folium.LayerControl(collapsed=False).add_to(mapa)
        
        path_html = os.path.join(output_dir, f'mapa_logistico_final_{datetime.now().strftime("%H%M%S")}.html')
        mapa.save(path_html)
        print(f'✅ Mapa listo en: {path_html}')

generar_btn.on_click(generar_mapa)
display(widgets.VBox([
    widgets.HBox([met_selector, ruta_selector]),
    generar_btn, 
    output_map
]))

In [ ]:
# ================== MAPA LOGÍSTICO PREMIUM: TOTAL DE CLIENTES ÚNICOS ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Configuración de rutas
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Inputs_mapa'
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
os.makedirs(output_dir, exist_ok=True)

archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))
archivo_matriz = archivos_excel[0] if archivos_excel else None

# 2. Leer datos
df_recorridos = pd.read_excel(archivo_matriz, sheet_name='Recorridos')

# Columnas
col_met, col_ruta, col_secuencia = '🏠 MET', '🛣️ Ruta', '🔢 Secuencia'
col_tipo, col_nombre, col_codigo = 'Tipo', '📚 Nombre', '🆔 Codigo'
col_lat, col_lon = '🌍 Latitud', '🌍 Longitud'

mets = sorted(df_recorridos[col_met].dropna().unique())
rutas = sorted(df_recorridos[col_ruta].dropna().unique())

# Widgets
met_selector = widgets.SelectMultiple(options=mets, value=tuple(mets), description='METs:')
ruta_selector = widgets.SelectMultiple(options=rutas, value=tuple(rutas), description='Rutas:')
generar_btn = widgets.Button(description='Generar Mapa Premium', button_style='success', layout={'width': '300px'})
output_map = widgets.Output()

def generar_mapa(b):
    with output_map:
        clear_output()
        mets_sel, rutas_sel = list(met_selector.value), list(ruta_selector.value)
        if not mets_sel or not rutas_sel: return
        
        # --- FILTRADO DE DATOS ---
        df_filtrado = df_recorridos[(df_recorridos[col_met].isin(mets_sel)) & (df_recorridos[col_ruta].isin(rutas_sel))]
        
        # --- LÓGICA DE CONTEO NO DUPLICADO ---
        # Filtramos solo los registros que son clientes
        df_solo_clientes = df_filtrado[df_filtrado[col_tipo] == 'Cliente']
        
        # CLAVE: nunique() cuenta códigos de cliente únicos aunque estén en varios METs
        total_clientes_reales = df_solo_clientes[col_codigo].nunique()
        
        # Conteo para popups individuales (aquí sí nos interesa por MET y Ruta)
        conteo_por_ruta = df_solo_clientes.groupby([col_met, col_ruta]).size().to_dict()

        mapa = folium.Map(location=[df_filtrado[col_lat].mean(), df_filtrado[col_lon].mean()], zoom_start=10)
        colores = ['#E63946', '#457B9D', '#2A9D8F', '#8338EC', '#F4A261', '#7209B7', '#0077B6', '#1D3557']
        color_idx = 0

        # --- CAPA FIJA MET ---
        fg_mets_fijos = folium.FeatureGroup(name="🏠 Ubicaciones MET", show=True)
        df_solo_mets = df_filtrado[df_filtrado[col_tipo] == 'MET'].drop_duplicates(subset=[col_met])
        for _, row in df_solo_mets.iterrows():
            icon_met = CustomIcon(icon_met_path, icon_size=(36,36)) if os.path.exists(icon_met_path) else folium.Icon(color='red', icon='home')
            folium.Marker([row[col_lat], row[col_lon]], icon=icon_met, tooltip=f"🏠 {row[col_met]}").add_to(fg_mets_fijos)
        fg_mets_fijos.add_to(mapa)

        # --- CAPAS DINÁMICAS (OCULTAS DE INICIO) ---
        for met in mets_sel:
            met_clean = str(met).replace('MET ', '').strip()
            df_met = df_filtrado[df_filtrado[col_met] == met]
            for ruta in sorted(df_met[col_ruta].unique()):
                capa_nombre = f"MET {met_clean} - Ruta {ruta}"
                fg = folium.FeatureGroup(name=capa_nombre, show=False)
                
                df_ruta = df_met[df_met[col_ruta] == ruta].sort_values(col_secuencia)
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                paradas_en_esta_ruta = conteo_por_ruta.get((met, ruta), 0)
                
                folium.PolyLine(list(zip(df_ruta[col_lat], df_ruta[col_lon])), color=color_ruta, weight=5, opacity=0.6).add_to(fg)
                
                for _, row in df_ruta.iterrows():
                    if row[col_tipo] != 'MET':
                        # Popup Premium
                        popup_html = f'''
                        <div style="font-family: 'Segoe UI', sans-serif; width: 220px; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 10px rgba(0,0,0,0.2); border: 1px solid #ddd;">
                            <div style="background: linear-gradient(135deg, {color_ruta} 0%, #333 100%); color: white; padding: 10px; text-align: center;">
                                <div style="font-size: 10px; opacity: 0.8; text-transform: uppercase;">Ruta {row[col_ruta]}</div>
                                <div style="font-size: 15px; font-weight: bold;">ID: {row[col_codigo]}</div>
                            </div>
                            <div style="padding: 12px; background: white;">
                                <div style="font-size: 12px; font-weight: 600; color: #333; margin-bottom: 5px;">{row[col_nombre]}</div>
                                <div style="display: flex; justify-content: space-between; font-size: 11px; color: #666; border-top: 1px solid #eee; padding-top: 8px;">
                                    <span><b>Posición:</b> {row[col_secuencia]}</span>
                                    <span><b>Paradas Ruta:</b> {paradas_en_esta_ruta}</span>
                                </div>
                            </div>
                        </div>
                        '''
                        icon_cli = DivIcon(html=f'''
                            <div style="font-size: 10px; color: white; background: {color_ruta}; border-radius: 50%; width: 24px; height: 24px; text-align: center; line-height: 24px; border: 2px solid #fff; font-weight: bold;">
                            {row[col_secuencia]}</div>''')
                        
                        folium.Marker([row[col_lat], row[col_lon]], icon=icon_cli, popup=folium.Popup(popup_html, max_width=250), tooltip=f"ID: {row[col_codigo]}").add_to(fg)
                fg.add_to(mapa)

        # --- TÍTULO CON CONTEO CORREGIDO ---
        mapa.get_root().html.add_child(folium.Element(f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9999; background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 1px solid rgba(255,255,255,0.3); min-width: 420px; font-family: 'Segoe UI', sans-serif;">
            <h2 style="margin: 0; text-align: center; font-size:16px; letter-spacing: 1px;">🗺️ ANÁLISIS DE RECORRIDOS</h2>
            <p style="margin: 4px 0 0 0; text-align: center; font-size: 11px; opacity: 0.9;">
                📍 METs: {len(mets_sel)} | 🛣️ Rutas: {len(rutas_sel)} | 👥 Clientes Únicos: {total_clientes_reales}
            </p>
        </div>
        '''))

        # --- CSS Y JS DEL PANEL DE CONTROL ---
        mapa.get_root().html.add_child(folium.Element('''
        <style>
            .leaflet-control-layers-list { padding: 10px; min-width: 280px; font-family: 'Segoe UI', sans-serif; border-radius: 8px; }
            .leaflet-control-layers-overlays { display: none !important; } 
            .layer-control-buttons { display: flex; gap: 5px; margin-bottom: 10px; border-bottom: 1px solid #eee; padding-bottom: 10px; }
            .layer-btn { flex: 1; padding: 6px; cursor: pointer; border: none; border-radius: 4px; font-weight: bold; font-size: 12px; }
            .select-all { background-color: #2a9d8f !important; color: white; }
            .deselect-all { background-color: #e63946 !important; color: white; }
            .met-buttons-row { display: flex; flex-wrap: wrap; gap: 4px; margin-bottom: 10px; }
            .met-btn { padding: 5px 10px; font-size: 11px; cursor: pointer; background: white; border: 1px solid #457b9d; color: #1d3557; border-radius: 4px; font-weight: bold; }
            .ruta-buttons-row { display: grid; grid-template-columns: repeat(4, 1fr); gap: 4px; }
            .ruta-btn { padding: 4px; font-size: 10px; cursor: pointer; background: white; border: 1px solid #f4a261; color: #e76f51; border-radius: 4px; font-weight: bold; }
        </style>
        '''))

        layer_control_js = r'''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            setTimeout(function() {
                const list = document.querySelector('.leaflet-control-layers-list');
                const overlays = document.querySelector('.leaflet-control-layers-overlays');
                if (!list || !overlays) return;
                const cbs = Array.from(overlays.querySelectorAll('input[type="checkbox"]'));
                const mets = new Set(); const rutas = new Set();
                cbs.forEach(cb => {
                    const txt = cb.parentElement.querySelector('span').textContent;
                    if (txt.includes("🏠 Ubicaciones MET")) return; 
                    const m = txt.match(/MET\s+([^\s-]+)/);
                    const r = txt.match(/Ruta\s+([^\s-]+)/);
                    if(m) mets.add(m[1]);
                    if(r) rutas.add(r[1]);
                });
                const container = document.createElement('div');
                let html = '<div class="layer-control-buttons"><button class="layer-btn select-all">✓ Todo</button><button class="layer-btn deselect-all">✗ Nada</button></div>';
                html += '<div class="met-buttons-row">';
                Array.from(mets).sort().forEach(m => { html += `<button class="met-btn">${m}</button>`; });
                html += '</div>';
                html += '<div class="ruta-buttons-row">';
                Array.from(rutas).sort((a,b) => a-b).forEach(r => { html += `<button class="ruta-btn">R${r}</button>`; });
                html += '</div>';
                container.innerHTML = html;
                list.insertBefore(container, overlays);
                container.addEventListener('click', (e) => {
                    const btn = e.target;
                    const txt = btn.textContent;
                    cbs.forEach(cb => {
                        const label = cb.parentElement.querySelector('span').textContent;
                        if (label.includes("🏠 Ubicaciones MET")) return;
                        if(btn.classList.contains('select-all')) { if(!cb.checked) cb.click(); }
                        else if(btn.classList.contains('deselect-all')) { if(cb.checked) cb.click(); }
                        else if(btn.classList.contains('met-btn')) {
                            const isMet = label.includes('MET ' + txt);
                            if(isMet && !cb.checked) cb.click(); else if(!isMet && cb.checked) cb.click();
                        }
                        else if(btn.classList.contains('ruta-btn')) {
                            const rNum = txt.replace('R', '');
                            const isRuta = label.endsWith('Ruta ' + rNum);
                            if(isRuta && !cb.checked) cb.click(); else if(!isRuta && cb.checked) cb.click();
                        }
                    });
                });
            }, 1000);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(layer_control_js))
        folium.LayerControl(collapsed=False).add_to(mapa)
        
        path_html = os.path.join(output_dir, f'mapa_final_corregido_{datetime.now().strftime("%H%M%S")}.html')
        mapa.save(path_html)
        print(f'✅ Mapa listo con conteo único: {path_html}')

generar_btn.on_click(generar_mapa)
display(widgets.VBox([widgets.HBox([met_selector, ruta_selector]), generar_btn, output_map]))